# 🛡️ Smart Helmet Guard - YOLOv8 安全帽检测训练 (Google Colab)

在 Colab 的免费 T4 GPU 上训练 YOLOv8 安全帽检测模型。

⚠️ 重要: 确保 Colab 运行时已切换到 GPU (菜单栏 → 运行时 → 更改运行时类型 → T4 GPU)

In [ ]:
# 🔧 Step 1: 安装依赖 (GPU 版)
print('正在安装 GPU 版 PyTorch + ultralytics...')
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q ultralytics
print('✅ 完成')

In [ ]:
# 🔍 Step 2: 检查 GPU
import torch
print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 名称: {torch.cuda.get_device_name(0)}')
    print(f'GPU 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️ GPU 不可用！')

---
## 📥 获取 Kaggle 数据集

使用 Kaggle API Token 直接下载数据集（服务器对服务器，约3-8分钟）。

In [ ]:
# 📂 Step 3: 配置 Kaggle API + 下载数据集
import os, json

# ⬇️ 把 Token 粘贴到下面引号里
KAGGLE_TOKEN = ''
# ⬆️ 粘贴到上面

if not KAGGLE_TOKEN:
    raise ValueError('请先设置 KAGGLE_TOKEN！')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': 'kaggle_user', 'key': KAGGLE_TOKEN}, f)
!chmod 600 /root/.kaggle/kaggle.json
!pip install -q kaggle
print('\n✅ Kaggle API 配置完成')

print('\n正在下载 Hard Hat Detection 数据集...')
!kaggle datasets download -d andrewmvd/hard-hat-detection -p /content/
print('\n✅ 数据集下载完成！')

In [ ]:
# 📦 Step 4: 解压数据集
import zipfile, os

zip_files = [f for f in os.listdir('/content/') if f.endswith('.zip')]
if not zip_files:
    raise FileNotFoundError('没找到 zip 文件！')

zip_path = f'/content/{zip_files[0]}'
print(f'解压文件: {zip_path}')

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/dataset_raw')
    print(f'✅ 解压完成，共 {len(z.namelist())} 个文件')

print('\n解压后目录结构:')
for item in os.listdir('/content/dataset_raw'):
    full = os.path.join('/content/dataset_raw', item)
    if os.path.isdir(full):
        print(f'  📁 {item}/')
        for sub in os.listdir(full):
            sub_full = os.path.join(full, sub)
            count = len(os.listdir(sub_full)) if os.path.isdir(sub_full) else ''
            label = f'      📁 {sub}/ ({count} 个文件)' if os.path.isdir(sub_full) else f'      📄 {sub}'
            print(label)
    else:
        print(f'  📄 {item}')

In [ ]:
# 🔄 Step 5: 找到图片和标注目录
import os

raw_dir = '/content/dataset_raw'
images_dir = None
annots_dir = None

for root, dirs, files in os.walk(raw_dir):
    for d in dirs:
        if 'image' in d.lower() and images_dir is None:
            images_dir = os.path.join(root, d)
        if ('annot' in d.lower() or 'xml' in d.lower()) and annots_dir is None:
            annots_dir = os.path.join(root, d)

print(f'图片目录: {images_dir}')
print(f'标注目录: {annots_dir}')

if images_dir:
    imgs = [f for f in os.listdir(images_dir) if f.endswith(('.png','.jpg','.jpeg'))][:3]
    print(f'示例图片: {imgs}')
if annots_dir:
    xmls = [f for f in os.listdir(annots_dir) if f.endswith('.xml')][:3]
    print(f'示例标注: {xmls}')

if images_dir is None or annots_dir is None:
    raise ValueError('无法确定图片/标注目录')

In [ ]:
# 🏷️ Step 6: 获取类别
import xml.etree.ElementTree as ET
import os

def get_classes(annots_dir):
    classes = set()
    for f in os.listdir(annots_dir):
        if f.endswith('.xml'):
            tree = ET.parse(os.path.join(annots_dir, f))
            root = tree.getroot()
            for obj in root.findall('object'):
                name = obj.find('name').text
                classes.add(name)
    return sorted(classes)

classes = get_classes(annots_dir)
print(f'检测到的类别: {classes}')
class_to_id = {c: i for i, c in enumerate(classes)}
print(f'类别->ID 映射: {class_to_id}')

In [ ]:
# 🔄 Step 7: VOC XML -> YOLO TXT 格式转换
from sklearn.model_selection import train_test_split
import shutil, os

def convert_xml_to_yolo(xml_path, class_to_id):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find('size')
    w = int(size.find('width').text)
    h = int(size.find('height').text)
    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        if name not in class_to_id:
            continue
        cls_id = class_to_id[name]
        bnd = obj.find('bndbox')
        xmin = int(bnd.find('xmin').text)
        ymin = int(bnd.find('ymin').text)
        xmax = int(bnd.find('xmax').text)
        ymax = int(bnd.find('ymax').text)
        xc = ((xmin + xmax) / 2) / w
        yc = ((ymin + ymax) / 2) / h
        bw = (xmax - xmin) / w
        bh = (ymax - ymin) / h
        lines.append(f'{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    return lines

yolo_dir = '/content/dataset'
for split in ['train', 'valid']:
    os.makedirs(f'{yolo_dir}/images/{split}', exist_ok=True)
    os.makedirs(f'{yolo_dir}/labels/{split}', exist_ok=True)

xml_files = [f for f in os.listdir(annots_dir) if f.endswith('.xml')]
print(f'标注文件总数: {len(xml_files)}')

train_files, valid_files = train_test_split(xml_files, test_size=0.2, random_state=42)
print(f'训练集: {len(train_files)} 张, 验证集: {len(valid_files)} 张')

def process_split(files, split):
    skipped = 0
    for xml_file in files:
        base = xml_file.replace('.xml', '')
        img_file = None
        for ext in ['.png', '.jpg', '.jpeg', '.PNG', '.JPG']:
            candidate = base + ext
            if os.path.exists(os.path.join(images_dir, candidate)):
                img_file = candidate
                break
        if img_file is None:
            skipped += 1
            continue
        shutil.copy(os.path.join(images_dir, img_file), f'{yolo_dir}/images/{split}/{img_file}')
        lines = convert_xml_to_yolo(os.path.join(annots_dir, xml_file), class_to_id)
        with open(f'{yolo_dir}/labels/{split}/{base}.txt', 'w') as f:
            f.write('\n'.join(lines))
    if skipped:
        print(f'  ⚠️ {split} 跳过 {skipped} 个')

process_split(train_files, 'train')
process_split(valid_files, 'valid')
print('✅ 转换完成！')

print(f"\n训练图片: {len(os.listdir(f'{yolo_dir}/images/train'))} 张")
print(f"验证图片: {len(os.listdir(f'{yolo_dir}/images/valid'))} 张")

In [ ]:
# 📄 Step 8: 生成 data.yaml
data_yaml = f'''path: /content/dataset
train: images/train
val: images/valid

nc: {len(classes)}
names: {classes}
'''

with open('/content/dataset/data.yaml', 'w') as f:
    f.write(data_yaml)

print(data_yaml)
print('✅ data.yaml 已生成')

In [ ]:
# 🚀 Step 9: 开始训练 YOLOv8 (GPU)
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=416,
    batch=16,
    patience=15,
    save=True,
    project='/content/runs',
    name='helmet_detect',
    exist_ok=True
)

print('🎉 训练完成！')

In [ ]:
# 📊 Step 10: 查看训练结果
import os
from IPython.display import Image as IPImage, display

results_dir = '/content/runs/helmet_detect'
best_model = f'{results_dir}/weights/best.pt'
print(f'最佳模型: {best_model}')
print(f"文件大小: {os.path.getsize(best_model) / 1024**2:.1f} MB")

for f in ['results.png', 'confusion_matrix.png', 'F1_curve.png']:
    path = f'{results_dir}/{f}'
    if os.path.exists(path):
        print(f"\n📈 {f}:")
        display(IPImage(path, width=600))

In [ ]:
# 💾 Step 11: 下载训练好的模型
from google.colab import files

best_model_path = '/content/runs/helmet_detect/weights/best.pt'
print('正在下载 best.pt...')
files.download(best_model_path)
print('\n✅ 下载完成！放到 F:/safety_helmet_detection/ 下')

In [ ]:
# 🧪 Step 12 (可选): 验证模型效果
from ultralytics import YOLO
import os

model = YOLO('/content/runs/helmet_detect/weights/best.pt')
metrics = model.val(data='/content/dataset/data.yaml')
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

valid_images = os.listdir('/content/dataset/images/valid')[:4]
for img in valid_images:
    results = model.predict(
        f'/content/dataset/images/valid/{img}',
        save=True, conf=0.25,
        project='/content/predictions',
        name='demo', exist_ok=True
    )
    print(f'预测完成: {img}')

print('\n预测结果在 /content/predictions/demo/')